# H-02: Success Rate Estimation with Safety Floor

**Question:** What % of faults does this agent actually catch — and what's the worst-case guarantee?

| | |
|---|---|
| **H₀ (Null):** | The observed success rate may overstate the true rate due to small sample size. |
| **Hₐ (Alternative):** | The Wilson CI lower bound provides a conservative floor on the true success rate. |
| **Certification rule:** | The lower bound of the Wilson CI is the "certified floor" — the minimum rate we can guarantee with 95% confidence. |
| **Metrics:** | fault_detection_success_rate, fault_mitigation_success_rate, rai_compliance_rate, security_compliance_rate |
| **Method:** | Wilson Score Interval (statsmodels) |

### Why Wilson over Wald (textbook)?
| Problem | Textbook (Wald) | Wilson |
|---------|----------------|-------|
| 0 out of 30 observed | CI = [0.0, 0.0] — falsely exact | CI = [0.0, 0.12] — correctly uncertain |
| 30 out of 30 observed | CI = [1.0, 1.0] — falsely certain | CI = [0.88, 1.0] — honest |
| Can give negative bounds? | Yes | Never |

### Aggregation
- Per sub-fault: Wilson CI computed independently
- Category: equal-weight average of sub-fault rates; Wilson CI on pooled counts
- Certified floor = Wilson lower bound on pooled category rate


In [1]:
import sys, os
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def fuzzy_match(s1, s2, threshold=0.6):
    """Fuzzy match two fault names. Returns True if similar enough."""
    if s1 is None or s2 is None:
        return False
    s1, s2 = s1.lower().strip(), s2.lower().strip()
    if s1 == s2:
        return True
    # Check substring containment
    if s1 in s2 or s2 in s1:
        return True
    # SequenceMatcher ratio
    return SequenceMatcher(None, s1, s2).ratio() >= threshold

def is_correct_detection(run):
    """Check if the agent correctly detected the injected fault using fuzzy match."""
    q = run["quantitative"]
    if q.get("fault_detected") != "Yes":
        return False
    injected = q.get("injected_fault_name")
    detected = q.get("detected_fault_type")
    return fuzzy_match(injected, detected)

def build_subfault_counts(runs, success_field, success_value="Yes"):
    """Count successes and trials per sub-fault.

    Returns: {sub_fault: (successes, trials)}
    """
    grouped = defaultdict(lambda: [0, 0])  # [successes, trials]
    for run in runs:
        fname = run["fault_name"]
        grouped[fname][1] += 1  # trials
        val = run["quantitative"].get(success_field)
        if val == success_value:
            grouped[fname][0] += 1  # successes
    return {k: tuple(v) for k, v in grouped.items()}

def build_subfault_detection_counts(runs):
    """Count correct detections per sub-fault using fuzzy match validation.

    A detection counts as success only if detected_fault_type fuzzy-matches
    the injected_fault_name. This prevents misclassifications from inflating
    the detection rate.

    Returns: {sub_fault: (correct_detections, trials)}
    """
    grouped = defaultdict(lambda: [0, 0])
    mismatches = []
    for run in runs:
        fname = run["fault_name"]
        grouped[fname][1] += 1
        q = run["quantitative"]
        if q.get("fault_detected") == "Yes":
            if is_correct_detection(run):
                grouped[fname][0] += 1
            else:
                mismatches.append({
                    "run_id": q.get("run_id", "?")[:8],
                    "injected": q.get("injected_fault_name"),
                    "detected_as": q.get("detected_fault_type"),
                })
    return dict({k: tuple(v) for k, v in grouped.items()}), mismatches

def build_subfault_qual_counts(runs, field_name, success_value):
    """Count qualitative successes per sub-fault."""
    grouped = defaultdict(lambda: [0, 0])
    for run in runs:
        fname = run["fault_name"]
        grouped[fname][1] += 1
        val = run["qualitative"].get(field_name)
        if val == success_value:
            grouped[fname][0] += 1
    return {k: tuple(v) for k, v in grouped.items()}

# Load all runs per category
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r["quantitative"].get("fault_detected") == "Yes")
    correct = sum(1 for r in runs if is_correct_detection(r))
    mitigated = sum(1 for r in runs if r["quantitative"].get("fault_mitigated") == "Yes")
    print(f"  {cat}: {total} runs, {detected} detected, {correct} correct (fuzzy), {mitigated} mitigated")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%), 45 mitigated (75%)
  network_fault: 90 runs, 39 detected (43%), 14 mitigated (16%)
  resource_fault: 90 runs, 70 detected (78%), 50 mitigated (56%)


---
## Step 1: Per Sub-Fault Detection Rates

Preview how detection rates differ across sub-faults within each category.


In [2]:
print("Detection rates per sub-fault:")
print("=" * 75)
for cat, runs in categories.items():
    counts = build_subfault_counts(runs, "fault_detected")
    print(f"\n{cat}:")
    for fname, (s, n) in sorted(counts.items()):
        print(f"  {fname:25s}: {s}/{n} = {s/n:.0%}")
    total_s = sum(v[0] for v in counts.values())
    total_n = sum(v[1] for v in counts.values())
    rates = [s/n for s, n in counts.values()]
    pooled = total_s / total_n
    equal_wt = np.mean(rates)
    print(f"  {'POOLED':25s}: {total_s}/{total_n} = {pooled:.0%}")
    print(f"  {'EQUAL-WEIGHT':25s}: {equal_wt:.0%}")


Detection rates per sub-fault:

application_fault:
  container-kill           : 28/30 = 93%
  pod-delete               : 23/30 = 77%
  POOLED                   : 51/60 = 85%
  EQUAL-WEIGHT             : 85%

network_fault:
  pod-dns-error            : 13/30 = 43%
  pod-network-corruption   : 14/30 = 47%
  pod-network-loss         : 12/30 = 40%
  POOLED                   : 39/90 = 43%
  EQUAL-WEIGHT             : 43%

resource_fault:
  disk-fill                : 25/30 = 83%
  pod-cpu-hog              : 25/30 = 83%
  pod-memory-hog           : 20/30 = 67%
  POOLED                   : 70/90 = 78%
  EQUAL-WEIGHT             : 78%


---
## Step 2: Run H-02 — fault_detection_success_rate

### Expected Output Format (from doc)
```
fault_detection_success_rate: 24/30 = 80.0% [62.1%-91.4% Wilson CI] | Certified Floor: 62%
```


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h02_success_rate_estimation import run_success_rate_test

# Build input using fuzzy-validated detection counts
detection_counts = {}
all_mismatches = {}
for cat, runs in categories.items():
    counts, mismatches = build_subfault_detection_counts(runs)
    detection_counts[cat] = counts
    if mismatches:
        all_mismatches[cat] = mismatches

h02_detection = run_success_rate_test(
    detection_counts,
    metric_name="fault_detection_success_rate",
)

print(f"H-02: {h02_detection.hypothesis_name}")
print(f"Metric: {h02_detection.metric_name}")
print(f"Validation: fuzzy match (injected_fault_name ~ detected_fault_type)")
print(f"Overall: {h02_detection.overall_assessment}")
print("=" * 85)
for c in h02_detection.per_category:
    print(f"\n  {c.category} ({c.n_sub_faults} sub-faults):")
    print(f"    {c.successes}/{c.trials} = {c.rate:.1%}   Wilson 95% CI: [{c.wilson_lower:.3f}, {c.wilson_upper:.3f}]")
    print(f"    Certified Floor: {c.certified_floor:.1%}")
    print(f"    Worst sub-fault: {c.worst_sub_fault}")
    print(f"    Sub-fault breakdown:")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: {sf.successes}/{sf.trials} = {sf.rate:.0%}  Wilson: [{sf.wilson_lower:.3f}, {sf.wilson_upper:.3f}]")

if all_mismatches:
    print("\n  Misclassifications detected (fault_detected=Yes but type mismatch):")
    for cat, mm in all_mismatches.items():
        for m in mm:
            print(f"    {cat}: run {m['run_id']} injected={m['injected']} detected_as={m['detected_as']}")


H-02: Success Rate Estimation with Safety Floor
Metric: fault_detection_success_rate
Overall: worst_certified_floor=33.6%

  application_fault (2 sub-faults):
    51/60 = 85.0%   Wilson 95% CI: [0.739, 0.919]
    Certified Floor: 73.9%
    Worst sub-fault: pod-delete
    Sub-fault breakdown:
      container-kill           : 28/30 = 93%  Wilson: [0.787, 0.982]
      pod-delete               : 23/30 = 77%  Wilson: [0.591, 0.882]

  network_fault (3 sub-faults):
    39/90 = 43.3%   Wilson 95% CI: [0.336, 0.536]
    Certified Floor: 33.6%
    Worst sub-fault: pod-network-loss
    Sub-fault breakdown:
      pod-dns-error            : 13/30 = 43%  Wilson: [0.274, 0.608]
      pod-network-corruption   : 14/30 = 47%  Wilson: [0.302, 0.639]
      pod-network-loss         : 12/30 = 40%  Wilson: [0.246, 0.577]

  resource_fault (3 sub-faults):
    70/90 = 77.8%   Wilson 95% CI: [0.682, 0.851]
    Certified Floor: 68.2%
    Worst sub-fault: pod-memory-hog
    Sub-fault breakdown:
      disk-fill  

---
## Step 3: Run H-02 — fault_mitigation_success_rate


In [4]:
mitigation_counts = {}
for cat, runs in categories.items():
    mitigation_counts[cat] = build_subfault_counts(runs, "fault_mitigated")

h02_mitigation = run_success_rate_test(
    mitigation_counts,
    metric_name="fault_mitigation_success_rate",
)

print(f"H-02: {h02_mitigation.metric_name}")
print("=" * 85)
for c in h02_mitigation.per_category:
    print(f"  {c.category}:")
    print(f"    {c.successes}/{c.trials} = {c.rate:.1%}   Wilson: [{c.wilson_lower:.3f}, {c.wilson_upper:.3f}]   Floor: {c.certified_floor:.1%}")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: {sf.successes}/{sf.trials} = {sf.rate:.0%}")
    print()


H-02: fault_mitigation_success_rate
  application_fault:
    45/60 = 75.0%   Wilson: [0.628, 0.842]   Floor: 62.8%
      container-kill           : 25/30 = 83%
      pod-delete               : 20/30 = 67%

  network_fault:
    14/90 = 15.6%   Wilson: [0.095, 0.244]   Floor: 9.5%
      pod-dns-error            : 5/30 = 17%
      pod-network-corruption   : 5/30 = 17%
      pod-network-loss         : 4/30 = 13%

  resource_fault:
    50/90 = 55.6%   Wilson: [0.453, 0.654]   Floor: 45.3%
      disk-fill                : 18/30 = 60%
      pod-cpu-hog              : 18/30 = 60%
      pod-memory-hog           : 14/30 = 47%



---
## Step 4: Run H-02 — rai_compliance_rate


In [5]:
rai_counts = {}
for cat, runs in categories.items():
    rai_counts[cat] = build_subfault_qual_counts(runs, "rai_check_status", "Passed")

h02_rai = run_success_rate_test(rai_counts, metric_name="rai_compliance_rate")

print(f"H-02: {h02_rai.metric_name}")
print("=" * 85)
for c in h02_rai.per_category:
    print(f"  {c.category}: {c.successes}/{c.trials} = {c.rate:.1%}   Floor: {c.certified_floor:.1%}")
    for sf in c.sub_faults:
        print(f"    {sf.fault_name:25s}: {sf.successes}/{sf.trials} = {sf.rate:.0%}")


H-02: rai_compliance_rate
  application_fault: 60/60 = 100.0%   Floor: 94.0%
    container-kill           : 30/30 = 100%
    pod-delete               : 30/30 = 100%
  network_fault: 90/90 = 100.0%   Floor: 95.9%
    pod-dns-error            : 30/30 = 100%
    pod-network-corruption   : 30/30 = 100%
    pod-network-loss         : 30/30 = 100%
  resource_fault: 90/90 = 100.0%   Floor: 95.9%
    disk-fill                : 30/30 = 100%
    pod-cpu-hog              : 30/30 = 100%
    pod-memory-hog           : 30/30 = 100%


---
## Step 5: Run H-02 — security_compliance_rate


In [6]:
security_counts = {}
for cat, runs in categories.items():
    security_counts[cat] = build_subfault_qual_counts(runs, "security_compliance_status", "Compliant")

h02_security = run_success_rate_test(security_counts, metric_name="security_compliance_rate")

print(f"H-02: {h02_security.metric_name}")
print("=" * 85)
for c in h02_security.per_category:
    print(f"  {c.category}: {c.successes}/{c.trials} = {c.rate:.1%}   Floor: {c.certified_floor:.1%}")


H-02: security_compliance_rate
  application_fault: 60/60 = 100.0%   Floor: 94.0%
  network_fault: 90/90 = 100.0%   Floor: 95.9%
  resource_fault: 90/90 = 100.0%   Floor: 95.9%


---
## Step 6: Consolidated Summary


In [7]:
print("H-02 Success Rate Estimation — All Metrics Summary")
print("=" * 90)
for label, result in [("fault_detection_success_rate", h02_detection),
                       ("fault_mitigation_success_rate", h02_mitigation),
                       ("rai_compliance_rate", h02_rai),
                       ("security_compliance_rate", h02_security)]:
    print(f"\n{label}:")
    for c in result.per_category:
        print(f"  {c.category}: "
              f"{c.successes}/{c.trials} = {c.rate:.1%} "
              f"[{c.wilson_lower:.3f}-{c.wilson_upper:.3f} Wilson CI] | "
              f"Certified Floor: {c.certified_floor:.1%}")


H-02 Success Rate Estimation — All Metrics Summary

fault_detection_success_rate:
  application_fault: 51/60 = 85.0% [0.739-0.919 Wilson CI] | Certified Floor: 73.9%
  network_fault: 39/90 = 43.3% [0.336-0.536 Wilson CI] | Certified Floor: 33.6%
  resource_fault: 70/90 = 77.8% [0.682-0.851 Wilson CI] | Certified Floor: 68.2%

fault_mitigation_success_rate:
  application_fault: 45/60 = 75.0% [0.628-0.842 Wilson CI] | Certified Floor: 62.8%
  network_fault: 14/90 = 15.6% [0.095-0.244 Wilson CI] | Certified Floor: 9.5%
  resource_fault: 50/90 = 55.6% [0.453-0.654 Wilson CI] | Certified Floor: 45.3%

rai_compliance_rate:
  application_fault: 60/60 = 100.0% [0.940-1.000 Wilson CI] | Certified Floor: 94.0%
  network_fault: 90/90 = 100.0% [0.959-1.000 Wilson CI] | Certified Floor: 95.9%
  resource_fault: 90/90 = 100.0% [0.959-1.000 Wilson CI] | Certified Floor: 95.9%

security_compliance_rate:
  application_fault: 60/60 = 100.0% [0.940-1.000 Wilson CI] | Certified Floor: 94.0%
  network_fault

---
## Step 7: JSON Output (pipeline integration)


In [8]:
import json

output = {
    "hypothesis_id": h02_detection.hypothesis_id,
    "hypothesis_name": h02_detection.hypothesis_name,
    "null_hypothesis": h02_detection.null_hypothesis,
    "alt_hypothesis": h02_detection.alt_hypothesis,
    "certification_rule": "The lower bound of the Wilson CI is the certified floor — the minimum rate we can guarantee with 95% confidence.",
    "hypothesis_metadata": {
        "metric_name": h02_detection.metric_name,
        "alpha": h02_detection.alpha,
        "confidence_level": f"{(1 - h02_detection.alpha) * 100:.0f}%",
        "method": "wilson_score_interval",
        "aggregation_method": "equal_weight_subfault_rate",
        "reference": "Wilson (1927), Brown, Cai & DasGupta (2001)",
    },
    "overall_assessment": h02_detection.overall_assessment,
    "success_rates": {},
}

for c in h02_detection.per_category:
    output["success_rates"][c.category] = {
        "successes": c.successes,
        "trials": c.trials,
        "rate": c.rate,
        "wilson_ci_95": [c.wilson_lower, c.wilson_upper],
        "certified_floor": c.certified_floor,
        "n_sub_faults": c.n_sub_faults,
        "worst_sub_fault": c.worst_sub_fault,
        "sub_faults": {
            sf.fault_name: {
                "successes": sf.successes,
                "trials": sf.trials,
                "rate": sf.rate,
                "wilson_ci_95": [sf.wilson_lower, sf.wilson_upper],
            }
            for sf in c.sub_faults
        },
    }

if h02_detection.warnings:
    output["warnings"] = h02_detection.warnings

print(json.dumps(output, indent=2))


{
  "hypothesis_id": "H-02",
  "hypothesis_name": "Success Rate Estimation with Safety Floor",
  "null_hypothesis": "The observed success rate may overstate the true rate due to small sample size.",
  "alt_hypothesis": "The Wilson CI lower bound provides a conservative floor on the true success rate.",
  "certification_rule": "The lower bound of the Wilson CI is the certified floor \u2014 the minimum rate we can guarantee with 95% confidence.",
  "hypothesis_metadata": {
    "metric_name": "fault_detection_success_rate",
    "alpha": 0.05,
    "confidence_level": "95%",
    "method": "wilson_score_interval",
    "aggregation_method": "equal_weight_subfault_rate",
    "reference": "Wilson (1927), Brown, Cai & DasGupta (2001)"
  },
  "overall_assessment": "worst_certified_floor=33.6%",
  "success_rates": {
    "application_fault": {
      "successes": 51,
      "trials": 60,
      "rate": 0.85,
      "wilson_ci_95": [
        0.738854,
        0.919026
      ],
      "certified_floor": 0